# Desarrollo

In [32]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pathlib import Path
from dotenv import load_dotenv


In [33]:
load_dotenv()


True

In [35]:
spark = (
    SparkSession.builder.appName("Spark Flights ETL").master("local[8]").getOrCreate()
)

spark


In [20]:
BASE_DIR = Path.cwd().parent

BASE_DIR


PosixPath('/home/cora/Documents/data-engineer/moduloVI/spark-ejercicios')

In [40]:
flights = spark.read.csv(
    str(BASE_DIR / "flights-jan-apr-2018.csv"), header=True, inferSchema=True
)

flights.limit(5).toPandas()


,Month,DayofMonth,DayOfWeek,FlightDate,Origin,OriginCity,Dest,DestCity,DepTime,DepDelay,...,CancellationCode,Diverted,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,1,14,7,2018-01-14,SYR,"Syracuse, NY",DTW,"Detroit, MI",NaN,NaN,...,B,0.0,NaN,NaN,374.0,NaN,NaN,NaN,NaN,NaN
1,1,3,3,2018-01-03,SYR,"Syracuse, NY",LGA,"New York, NY",1348.0,-10.0,...,None,0.0,78.0,42.0,198.0,NaN,NaN,NaN,NaN,NaN
2,1,6,6,2018-01-06,SYR,"Syracuse, NY",LGA,"New York, NY",1410.0,12.0,...,None,0.0,93.0,45.0,198.0,12.0,0.0,12.0,0.0,0.0
3,1,7,7,2018-01-07,SYR,"Syracuse, NY",LGA,"New York, NY",1347.0,-11.0,...,None,0.0,68.0,38.0,198.0,NaN,NaN,NaN,NaN,NaN
4,1,8,1,2018-01-08,SYR,"Syracuse, NY",LGA,"New York, NY",1350.0,-8.0,...,None,0.0,79.0,39.0,198.0,NaN,NaN,NaN,NaN,NaN


In [44]:
import json

with open(BASE_DIR / "config/config.json") as f:
    properties = json.load(f)

print(properties)

flights.filter(F.col("FlightDate") == properties["raw_ingestion_date"]).groupBy(
    "FlightDate"
).count().orderBy(F.desc("count")).toPandas()


{'DATABRICKS_CONFIG_PROFILE': 'DATAENGINEERING', 'EXECUTION_ENVIRONMENT': 'databricks', 'STORAGE_ACCOUNT_NAME': 'masteravdc001sta', 'raw_input_file': 'abfss://datos@masteravdc001sta.dfs.core.windows.net/flights-jan-apr-2018.csv', 'raw_ingestion_date': '2018-04-20', 'bronze_table': 'masteravdc001dbr.bronze.flights', 'silver_table': 'masteravdc001dbr.silver.flights'}


,FlightDate,count
0,2018-04-20,22677


In [37]:
flights.select(
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"
).limit(5).toPandas()


,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0.0,0.0,16.0,0.0,0.0
1,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN


### Reemplazar lo nulos por 0 en las clolumnas CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay

In [23]:
flights.fillna(
    0,
    subset=[
        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay",
    ],
).select(
    "CarrierDelay", "WeatherDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"
).limit(5).toPandas()


,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,12.0,0.0,12.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


### Convertir la columna Diverted en booleana

In [24]:
flights.select("Diverted").groupBy("Diverted").count().toPandas()


,Diverted,count
0,0.0,2497670
1,1.0,5443


In [25]:
flights.withColumn("Diverted", F.col("Diverted") == 1).select("Diverted").groupBy(
    "Diverted"
).count().toPandas()


,Diverted,count
0,True,5443
1,False,2497670


### Crear la columna FlightTs

In [26]:
f"{flights.where(F.col('DepTime').isNull()).count():,}"


'54,903'

In [27]:
deptime_count = flights.groupBy("DepTime").count()
min_max_deptime = deptime_count.agg(
    F.min("DepTime").alias("min_deptime"), F.max("DepTime").alias("max_deptime")
).collect()[0]

deptime_count.where(
    F.col("DepTime").isin(
        [min_max_deptime["min_deptime"], min_max_deptime["max_deptime"]]
    )
).toPandas()


,DepTime,count
0,1,211
1,2400,164


Primero es necesario rellenar los valores nulos, en este caso por comodidad con **0**. `to_timestamp` en el formato **HHmm** no interpreta **2400** se mapea a **0**.


In [28]:
# flights \
#     .where("DepTime = 2400") \
#     .withColumn(
#     "FlightTs",
#     F.to_timestamp(
#         F.concat_ws(" ", F.col("FlightDate"), F.lpad(F.col("DepTime"), 4, "0")),
#         "yyyy-MM-dd HHmm",
#     ),
# ).select("DepTime", "FlightDate", "FlightTs").limit(5).toPandas()

# org.apache.spark.SparkDateTimeException: [CANNOT_PARSE_TIMESTAMP] Text '2018-01-01 2400' could not be parsed: Invalid value for HourOfDay (valid values 0 - 23): 24.
# Use `try_to_timestamp` to tolerate invalid input string and return NULL instead. SQLSTATE: 22007


In [29]:
flights.fillna(0, subset=["DepTime"]).withColumn(
    "DepTime", F.when(F.col("DepTime") == 2400, 0).otherwise(F.col("DepTime"))
).withColumn(
    "FlightTs",
    F.to_timestamp(
        F.concat_ws(" ", F.col("FlightDate"), F.lpad(F.col("DepTime"), 4, "0")),
        "yyyy-MM-dd HHmm",
    ),
).select("DepTime", "FlightDate", "FlightTs").limit(5).toPandas()


,DepTime,FlightDate,FlightTs
0,0,2018-01-14,2018-01-14 00:00:00
1,1348,2018-01-03,2018-01-03 13:48:00
2,1410,2018-01-06,2018-01-06 14:10:00
3,1347,2018-01-07,2018-01-07 13:47:00
4,1350,2018-01-08,2018-01-08 13:50:00


In [38]:
from spark_flights_etl.etl import Featurizer


In [39]:
Featurizer.preprocesa(flights.limit(5)).toPandas()


,Month,DayofMonth,DayOfWeek,FlightDate,Origin,OriginCity,Dest,DestCity,DepTime,DepDelay,...,Diverted,ActualElapsedTime,AirTime,Distance,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,FlightTs
0,4,8,7,2018-04-08,MSP,"Minneapolis, MN",BJI,"Bemidji, MN",2200,-5.0,...,False,85.0,37.0,199.0,0.0,0.0,16.0,0.0,0.0,2018-04-08 22:00:00
1,4,23,1,2018-04-23,PHX,"Phoenix, AZ",ORD,"Chicago, IL",715,-5.0,...,False,196.0,176.0,1440.0,0.0,0.0,0.0,0.0,0.0,2018-04-23 07:15:00
2,3,28,3,2018-03-28,DFW,"Dallas/Fort Worth, TX",MLU,"Monroe, LA",1429,-6.0,...,False,72.0,48.0,293.0,0.0,0.0,0.0,0.0,0.0,2018-03-28 14:29:00
3,3,17,6,2018-03-17,ECP,"Panama City, FL",IAH,"Houston, TX",1424,-8.0,...,False,134.0,110.0,572.0,0.0,0.0,0.0,0.0,0.0,2018-03-17 14:24:00
4,4,27,5,2018-04-27,LGB,"Long Beach, CA",SMF,"Sacramento, CA",1644,4.0,...,False,74.0,61.0,387.0,0.0,0.0,0.0,0.0,0.0,2018-04-27 16:44:00
